# 01 — Basic stacking on a synthetic dataset

Minimal end-to-end example: 2 embeddings → stacking ensemble → predict. Uses the same tiny synthetic dataset as our test suite (no GPU needed).

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
from scboost_ensemble import StackingEnsembleDonorMetaCached

## Build a tiny synthetic AnnData

5 donors × 4 cell types × 20 cells = 400 cells. Two synthetic 8-d embeddings where each cell type is centered at a distinct location.

In [ ]:
rng = np.random.default_rng(42)
n_donors, n_types, n_per = 5, 4, 20
n = n_donors * n_types * n_per
donor_ids   = np.repeat(np.arange(n_donors), n_types * n_per)
cell_types  = np.tile(np.repeat(np.arange(n_types), n_per), n_donors)
cell_type_names = np.array(['T cell', 'B cell', 'Macrophage', 'Endothelial'])[cell_types]
donor_names = np.array([f'donor{i}' for i in donor_ids])
centers_A = rng.normal(0, 1, size=(n_types, 8))
centers_B = rng.normal(0, 1, size=(n_types, 8))
emb_A = centers_A[cell_types] + rng.normal(0, 0.3, size=(n, 8))
emb_B = centers_B[cell_types] + rng.normal(0, 0.3, size=(n, 8))
adata = ad.AnnData(
    X=rng.random((n, 10), dtype=np.float32),
    obs=pd.DataFrame({'cell_type': cell_type_names, 'batch': donor_names,
                       'donor_id': donor_names},
                      index=[f'cell_{i:04d}' for i in range(n)]),
    obsm={'embA': emb_A.astype(np.float32), 'embB': emb_B.astype(np.float32)},
)
adata

## Split donor4 as test

In [ ]:
adata.obs['split'] = np.where(adata.obs['donor_id'] == 'donor4', 'test', 'train')
train = adata[adata.obs['split'] == 'train'].copy()
test  = adata[adata.obs['split'] == 'test'].copy()
print(f'train={train.n_obs}, test={test.n_obs}')

## Fit donor-aware stacking ensemble (CPU, 4 folds)

In [ ]:
stacker = StackingEnsembleDonorMetaCached(
    obsm_keys=['embA', 'embB'],
    label_key='cell_type', batch_key='batch',
    cv_scheme='donor', n_folds=4,
    early_stopping_rounds=5,
    device='cpu', n_gpus=1,
    use_batch_feature=False,
)
stacker.fit(train)

## Predict on the held-out donor

In [ ]:
preds = stacker.predict(test)
from sklearn.metrics import accuracy_score
acc = accuracy_score(test.obs['cell_type'], preds)
print(f'Held-out donor accuracy: {acc:.3f}')
print('Method importance:', stacker.get_method_importance())